[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/SFM/blob/main/Quantlets/Ch_12/SFM_ch12_reject_fairness/SFM_ch12_reject_fairness.ipynb)

# SFM_ch12_reject_fairness

Taiwan credit card data: a reject-inference experiment (an old score accepts the best 70%; a new logit trained on the accepted only against one trained on all applicants, both evaluated on all test applicants) and group fairness at a 75% approval cut-off (approval rates, approval of good payers, default rates and mean PDs by sex and age group; sex and age are not model inputs).

*Statistics of Financial Markets (SFM), Chapter 12: Scoring models. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/SFM/Chapter_12'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import sfm_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/SFM/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'          # last day of the saved data

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the SFM repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years

## Analysis

In [ ]:
CREDIT_RAW = 'https://raw.githubusercontent.com/danpele/SFM/main/data/credit/'

FILES = {'sgc': 'south_german_credit.csv', 'taiwan': 'taiwan_credit_card_default.csv'}

SEED = 2026

TEST_SHARE = 0.3

POP_BAD = 0.05

COST_FN = 5.0

COST_FP = 1.0

PDO, BASE_SCORE, BASE_ODDS = 20.0, 600.0, 50.0

IV_MIN = 0.1

N_REPEAT, N_FOLD = 20, 5

BINS = {'duration': [12, 18, 24, 36, np.inf], 'amount': [1300, 2000, 3000, 5000, np.inf], 'age': [25, 30, 35, 45, np.inf]}

NUMERIC = ['duration', 'amount', 'age']

SGC_VARS = ['status', 'duration', 'credit_history', 'purpose', 'amount', 'savings', 'employment_duration', 'installment_rate', 'personal_status_sex', 'other_debtors', 'present_residence', 'property', 'age', 'other_installment_plans', 'housing', 'number_credits', 'job', 'people_liable', 'telephone', 'foreign_worker']

STATUS = {1: 'no checking account', 2: 'balance < 0 DM', 3: '0 to 200 DM', 4: '>= 200 DM or salary account'}

TW_FEATURES = ['log_limit', 'age', 'pay0_m2', 'pay0_m1', 'pay0_1', 'pay0_2', 'pay0_3', 'late_months', 'util', 'log_pay1', 'log_pay2', 'log_pay3']

COLORS = {'good': '#1A3A6E', 'bad': '#CD0000', 'logit': '#CD0000', 'lda': '#1A3A6E', 'single': '#B5853F', 'small': '#2E7D32', 'random': '#8E44AD', 'perfect': '#E67E22'}

def load_credit(name='sgc'):
    """A credit data set from data/credit (local copy) or from the raw GitHub URL; adds 'default' (1 = bad)."""
    fn = FILES[name]
    local = next((os.path.join(d, 'data', 'credit', fn) for d in ('.', '..', '../..', '../../..')
                  if os.path.exists(os.path.join(d, 'data', 'credit', fn))), None)
    df = pd.read_csv(local or CREDIT_RAW + fn)
    if name == 'sgc':
        df['default'] = 1 - df['credit_risk']
    return df


def taiwan_features(df):
    """Model inputs of the Taiwan data: log limit, age, the repayment status of September 2005 as indicators (-2 no
    consumption, -1 paid in full, 1, 2, 3+ months late; 0 = minimum paid is the reference), the number of late months
    April-August, utilisation (bill / limit, clipped to [-1, 3]) and log payments of the last three months."""
    X = pd.DataFrame(index=df.index)
    X['log_limit'] = np.log(df['LIMIT_BAL'])
    X['age'] = df['AGE']
    p0 = df['PAY_0']
    X['pay0_m2'] = (p0 == -2).astype(int)
    X['pay0_m1'] = (p0 == -1).astype(int)
    X['pay0_1'] = (p0 == 1).astype(int)
    X['pay0_2'] = (p0 == 2).astype(int)
    X['pay0_3'] = (p0 >= 3).astype(int)
    X['late_months'] = sum((df[c] >= 1).astype(int) for c in ['PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6'])
    X['util'] = (df['BILL_AMT1'] / df['LIMIT_BAL']).clip(-1, 3)
    for i in (1, 2, 3):
        X[f'log_pay{i}'] = np.log1p(df[f'PAY_AMT{i}'].clip(lower=0))
    return X


def stratified_split(y, share=TEST_SHARE, seed=SEED):
    """Indices of a stratified train/test split (the same share of bads in both samples)."""
    rng = np.random.default_rng(seed)
    y = np.asarray(y)
    test = []
    for c in (0, 1):
        idx = np.flatnonzero(y == c)
        rng.shuffle(idx)
        test.append(idx[:int(round(share * len(idx)))])
    test = np.sort(np.concatenate(test))
    train = np.setdiff1d(np.arange(len(y)), test)
    return train, test


def logit_fit(X, y):
    """Logistic regression by maximum likelihood (Newton-Raphson): ln(p/(1-p)) = b0 + x'b.
    Returns the coefficients (constant first), standard errors and the log-likelihood."""
    X = np.column_stack([np.ones(len(X)), np.asarray(X, float)])
    y = np.asarray(y, float)
    b = np.zeros(X.shape[1])
    for _ in range(100):
        p = 1 / (1 + np.exp(-X @ b))
        W = p * (1 - p)
        H = X.T @ (X * W[:, None])
        step = np.linalg.solve(H, X.T @ (y - p))
        b += step
        if np.max(np.abs(step)) < 1e-10:
            break
    p = 1 / (1 + np.exp(-X @ b))
    H = X.T @ (X * (p * (1 - p))[:, None])
    se = np.sqrt(np.diag(np.linalg.inv(H)))
    ll = float(np.sum(y * np.log(p) + (1 - y) * np.log(1 - p)))
    return {'b': b, 'se': se, 'll': ll, 'n': len(y)}


def logit_predict(fit, X):
    X = np.column_stack([np.ones(len(X)), np.asarray(X, float)])
    return 1 / (1 + np.exp(-X @ fit['b']))


def lda_fit(X, y):
    """Fisher's linear discriminant: w = S_W^{-1}(m_1 - m_0) maximises the between/within variance ratio of w'x;
    with Normal classes and a common covariance, ln odds(bad | x) = c + w'x (the prior enters only c)."""
    X = np.asarray(X, float)
    y = np.asarray(y)
    m0, m1 = X[y == 0].mean(0), X[y == 1].mean(0)
    n0, n1 = (y == 0).sum(), (y == 1).sum()
    S = ((X[y == 0] - m0).T @ (X[y == 0] - m0) + (X[y == 1] - m1).T @ (X[y == 1] - m1)) / (n0 + n1 - 2)
    w = np.linalg.solve(S, m1 - m0)
    pi1 = n1 / (n0 + n1)
    c = -0.5 * (m1 + m0) @ w + np.log(pi1 / (1 - pi1))
    return {'w': w, 'c': float(c), 'm0': m0, 'm1': m1, 'S': S, 'pi1': float(pi1)}


def lda_predict(fit, X):
    """Posterior probability of default from the LDA log-odds."""
    return 1 / (1 + np.exp(-(fit['c'] + np.asarray(X, float) @ fit['w'])))


def auc(score, y):
    """AUC = P(score of a random bad > score of a random good) (ties count 1/2), via the Mann-Whitney ranks."""
    score, y = np.asarray(score, float), np.asarray(y)
    r = stats.rankdata(score)
    n1, n0 = (y == 1).sum(), (y == 0).sum()
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


def delong(score, y, score2=None):
    """DeLong et al. (1988): standard error of the AUC; with a second score on the same cases, the test of
    equal AUCs (z statistic and two-sided p-value)."""
    def comps(s):
        s = np.asarray(s, float)
        pos, neg = s[np.asarray(y) == 1], s[np.asarray(y) == 0]
        v10 = np.array([np.mean((p > neg) + 0.5 * (p == neg)) for p in pos])
        v01 = np.array([np.mean((pos > q) + 0.5 * (pos == q)) for q in neg])
        return v10, v01
    a10, a01 = comps(score)
    A = a10.mean()
    if score2 is None:
        var = np.var(a10, ddof=1) / len(a10) + np.var(a01, ddof=1) / len(a01)
        return {'auc': float(A), 'se': float(np.sqrt(var)), 'lo': float(A - 1.96 * np.sqrt(var)),
                'hi': float(A + 1.96 * np.sqrt(var))}
    b10, b01 = comps(score2)
    S10 = np.cov(np.vstack([a10, b10]))
    S01 = np.cov(np.vstack([a01, b01]))
    var = (S10[0, 0] + S10[1, 1] - 2 * S10[0, 1]) / len(a10) + (S01[0, 0] + S01[1, 1] - 2 * S01[0, 1]) / len(a01)
    z = (A - b10.mean()) / np.sqrt(var)
    return {'auc1': float(A), 'auc2': float(b10.mean()), 'z': float(z), 'p': float(2 * stats.norm.sf(abs(z)))}


def roc_points(score, y):
    """ROC curve: (FPR, TPR) when every score value is used as the cut-off (a higher score = riskier)."""
    score, y = np.asarray(score, float), np.asarray(y)
    thr = np.r_[np.inf, np.unique(score)[::-1]]
    tpr = np.array([np.mean(score[y == 1] >= t) for t in thr])
    fpr = np.array([np.mean(score[y == 0] >= t) for t in thr])
    return fpr, tpr, thr


def cap_points(score, y):
    """CAP (cumulative accuracy profile): share of all bads among the riskiest x% of the applicants."""
    score, y = np.asarray(score, float), np.asarray(y)
    order = np.argsort(-score, kind='mergesort')
    x = np.r_[0, np.arange(1, len(y) + 1) / len(y)]
    hit = np.r_[0, np.cumsum(y[order]) / y.sum()]
    return x, hit


def accuracy_ratio(score, y):
    """AR = (area between the model CAP and the diagonal) / (area between the perfect CAP and the diagonal);
    for a continuous score AR = Gini = 2 AUC - 1."""
    x, hit = cap_points(score, y)
    a_model = np.trapezoid(hit, x) - 0.5
    pbad = np.mean(y)
    a_perf = (1 - pbad / 2) - 0.5
    return float(a_model / a_perf)


def ks_stat(score, y):
    """Kolmogorov-Smirnov: the largest distance between the score distributions of the bads and the goods."""
    score, y = np.asarray(score, float), np.asarray(y)
    s = np.sort(np.unique(score))
    F1 = np.searchsorted(np.sort(score[y == 1]), s, side='right') / (y == 1).sum()
    F0 = np.searchsorted(np.sort(score[y == 0]), s, side='right') / (y == 0).sum()
    i = int(np.argmax(np.abs(F0 - F1)))
    return {'ks': float(np.abs(F0 - F1)[i]), 'at': float(s[i]), 's': s, 'F1': F1, 'F0': F0}


def brier(p, y):
    """Brier score: the mean squared difference between the predicted PD and the outcome (0 or 1)."""
    return float(np.mean((np.asarray(p) - np.asarray(y)) ** 2))


def calibration_table(p, y, k=10):
    """Mean predicted PD and observed default rate in k groups of equal size (sorted by PD); Hosmer-Lemeshow."""
    d = pd.DataFrame({'p': np.asarray(p), 'y': np.asarray(y)}).sort_values('p').reset_index(drop=True)
    d['g'] = np.arange(len(d)) * k // len(d)
    t = d.groupby('g').agg(n=('y', 'size'), pd_mean=('p', 'mean'), rate=('y', 'mean'), bad=('y', 'sum'))
    exp = t['n'] * t['pd_mean']
    hl = float((((t['bad'] - exp) ** 2) / (exp * (1 - t['pd_mean']))).sum())
    return t, hl, float(stats.chi2.sf(hl, k - 2))


def confusion(p, y, cut):
    """Reject (predict bad) if PD > cut. TP: bad rejected; FN: bad accepted; FP: good rejected; TN: good accepted."""
    p, y = np.asarray(p), np.asarray(y)
    pred = (p > cut).astype(int)
    TP = int(np.sum((pred == 1) & (y == 1)))
    FN = int(np.sum((pred == 0) & (y == 1)))
    FP = int(np.sum((pred == 1) & (y == 0)))
    TN = int(np.sum((pred == 0) & (y == 0)))
    n = len(y)
    return {'cut': float(cut), 'TP': TP, 'FN': FN, 'FP': FP, 'TN': TN, 'acc': (TP + TN) / n,
            'tpr': TP / (TP + FN), 'tnr': TN / (TN + FP), 'fpr': FP / (TN + FP), 'precision': TP / max(TP + FP, 1),
            'cost': (COST_FN * FN + COST_FP * FP) / n, 'reject': (TP + FP) / n}


def taiwan_model(df=None):
    """Logit on the Taiwan data (70/30 stratified split); sex, education and marital status are not inputs."""
    df = load_credit('taiwan') if df is None else df
    X = taiwan_features(df)
    y = df['default'].values
    tr, te = stratified_split(y)
    mu, sd = X.iloc[tr].mean(), X.iloc[tr].std()
    Z = (X - mu) / sd
    f = logit_fit(Z.iloc[tr], y[tr])
    return {'df': df, 'X': X, 'Z': Z, 'y': y, 'tr': tr, 'te': te, 'fit': f, 'p': logit_predict(f, Z)}


def reject_inference_sim(T, accept=0.7):
    """The bank accepted only the 70% best applicants of an old score (late payment in the last month and the limit);
    outcomes are known only for them. A new logit trained on the accepted (inputs that never vary among them, such as
    being late last month, cannot be estimated and are dropped) is compared with one trained on all applicants
    (unknown in practice), on the test applicants: all of them (through the door) and the accepted only."""
    Z, y, tr, te = T['Z'], T['y'], T['tr'], T['te']
    OLD = ['pay0_1', 'pay0_2', 'pay0_3', 'log_limit']
    old = logit_fit(Z.iloc[tr][OLD], y[tr])
    po = logit_predict(old, Z[OLD])
    cut = np.quantile(po[tr], accept)
    acc_tr = tr[po[tr] <= cut]
    acc_te = te[po[te] <= cut]
    rej_te = np.setdiff1d(te, acc_te)
    keep = [c for c in Z.columns if Z.iloc[acc_tr][c].std() > 1e-8]
    f_acc = logit_fit(Z.iloc[acc_tr][keep], y[acc_tr])
    pa, pf = logit_predict(f_acc, Z[keep]), T['p']
    return {'cut': float(cut), 'n_acc_train': int(len(acc_tr)), 'bad_acc': float(y[acc_tr].mean()),
            'bad_rej': float(y[np.setdiff1d(tr, acc_tr)].mean()), 'bad_all': float(y[tr].mean()),
            'dropped': [c for c in Z.columns if c not in keep],
            'auc_acc_model_on_acc': auc(pa[acc_te], y[acc_te]), 'auc_acc_model_on_all': auc(pa[te], y[te]),
            'auc_all_model_on_all': auc(pf[te], y[te]), 'auc_all_model_on_acc': auc(pf[acc_te], y[acc_te]),
            'auc_old_on_all': auc(po[te], y[te]), 'mean_pd_rej_acc_model': float(pa[rej_te].mean()),
            'mean_pd_rej_all_model': float(pf[rej_te].mean()), 'rate_rejected': float(y[rej_te].mean()),
            'rate_accepted_test': float(y[acc_te].mean())}


def fairness_table(T, approve=0.75):
    """Approve the 75% of test clients with the lowest PD; by sex (1 male, 2 female) and age group: default rate,
    mean PD, approval rate, and the share of non-defaulters approved (equal opportunity)."""
    df, y, p, te = T['df'], T['y'], T['p'], T['te']
    cut = np.quantile(p[te], approve)
    d = pd.DataFrame({'y': y[te], 'p': p[te], 'ok': p[te] <= cut, 'sex': df['SEX'].values[te],
                      'age': pd.cut(df['AGE'].values[te], [0, 25, 35, 50, 100], labels=['<= 25', '26-35', '36-50', '> 50'])})
    rows = {}
    for col, lab in [('sex', {1: 'men', 2: 'women'}), ('age', None)]:
        for g, s in d.groupby(col, observed=True):
            rows[lab[g] if lab else f'age {g}'] = {'n': int(len(s)), 'rate': float(s['y'].mean()), 'pd': float(s['p'].mean()),
                                                   'approve': float(s['ok'].mean()),
                                                   'tpr_good': float(s.loc[s['y'] == 0, 'ok'].mean())}
    return {'cut': float(cut), 'groups': rows}


def fig_fairness(F, save=True):
    """Observed default rate, mean PD and approval rate by sex and age group (Taiwan test sample)."""
    g = F['groups']
    labs = list(g)
    x = np.arange(len(labs))
    fig, ax = plt.subplots(figsize=(10, 4.0))
    w = 0.26
    ax.bar(x - w, [100 * g[k]['rate'] for k in labs], w, color=st.IDAred, label='observed default rate')
    ax.bar(x, [100 * g[k]['pd'] for k in labs], w, color=st.Amber, label='mean predicted PD')
    ax.bar(x + w, [100 * g[k]['approve'] for k in labs], w, color=st.MainBlue, label='approved (cut-off: best 75%)')
    ax.set_xticks(x)
    ax.set_xticklabels([f'{k}\n(n = {g[k]["n"]})' for k in labs])
    ax.set_ylabel('%')
    ax.set_ylim(0, 100)
    st.legend_outside_bottom(ax, ncol=3, y=-0.24)
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch12_fairness')

## Run

In [ ]:
T = taiwan_model()
print(reject_inference_sim(T))
F = fairness_table(T)
print(pd.DataFrame(F['groups']).T.round(3))
fig_fairness(F)

![sfm_ch12_fairness](./sfm_ch12_fairness.png)

Output: `sfm_ch12_fairness.pdf`